# Can we predict `passed_quiz`?
Question: from session-level information, can we predict whether a student passes the quiz that closes a session?

Plan: inspect the data, choose an honest validation scheme, compare against a no-skill baseline, then check what drives the signal.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold, KFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
RS = 0
d = pd.read_csv('data.csv')
print(pd.read_csv('columns.csv').to_string(index=False))
d.head()

## 1. Data checks

In [ ]:
print(d.shape, 'missing:', int(d.isna().sum().sum()), 'duplicate rows:', int(d.duplicated().sum()))
print('students:', d.student_key.nunique(), '| sessions/student:', d.groupby('student_key').size().describe()[['min','50%','max']].to_dict())
print('pass rate:', round(d.passed_quiz.mean(), 3))
d.describe().T

Things to keep in mind:
- **Several sessions per student** (1 to 17), so rows are not independent.
- `enrol_year` and `school_code` are constant within a student, but `prior_score` is not, even though a placement test should be taken once. It is stored per session record and carries per-session variation. A few values exceed 100.
- Pass rate seems to differ between students; checked next.

In [ ]:
g = d.groupby('student_key')
print('within-student SD of prior_score: mean', round(g.prior_score.std().mean(), 2), 'vs overall SD', round(d.prior_score.std(), 2))
print('rows with prior_score>100:', int((d.prior_score > 100).sum()))
sp = g.passed_quiz.agg(['mean', 'size']); print('SD of student-level pass rate:', round(sp['mean'].std(), 3))
rng = np.random.default_rng(RS)
sim = pd.Series([rng.binomial(n, d.passed_quiz.mean()) / n for n in sp['size']]).std()
print('SD expected if all students were identical (same session counts):', round(sim, 3))

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
d.groupby('hints_used').passed_quiz.mean().plot(marker='o', ax=ax[0], title='pass rate by hints_used')
d.groupby(pd.qcut(d.session_minutes, 10)).passed_quiz.mean().plot(marker='o', ax=ax[1], title='by session_minutes decile'); ax[1].set_xticklabels([])
d.groupby(pd.qcut(d.prior_score, 10)).passed_quiz.mean().plot(marker='o', ax=ax[2], title='by prior_score decile'); ax[2].set_xticklabels([])
for a in ax: a.set_ylim(0, 0.7)
plt.tight_layout()

## 2. Validation design
The aim is to predict for students we haven't seen, so I use **GroupKFold by `student_key`**: no student is in both train and test. `student_key` is not a feature. Section 4 shows how much a naive random split would flatter results.

Metrics: ROC AUC (ranking), PR AUC, log loss and Brier (probability quality). Baseline = predict the training pass rate. It predicts a constant per fold, so its pooled AUC is meaningless (0.5 within each fold; the pooled value dips below 0.5 because folds with a higher test pass rate get a lower constant). Use log loss / Brier to compare against it.

In [ ]:
feat = ['enrol_year', 'school_code', 'prior_score', 'session_minutes', 'hints_used']
X, y, grp = d[feat], d.passed_quiz, d.student_key
gkf = GroupKFold(n_splits=5)
gbm = lambda **k: HistGradientBoostingClassifier(random_state=RS, **k)
models = {
    'baseline (prior rate)': DummyClassifier(strategy='prior'),
    'logistic (minutes, hints)': make_pipeline(ColumnTransformer([('s', StandardScaler(), ['session_minutes', 'hints_used'])]), LogisticRegression()),
    'logistic (all)': make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    'grad. boosting (all)': gbm(max_depth=3, learning_rate=0.05, max_iter=200),
}
def score(p): return dict(AUC=roc_auc_score(y, p), PR_AUC=average_precision_score(y, p), logloss=log_loss(y, p), brier=brier_score_loss(y, p))
oof = {n: cross_val_predict(mod, X, y, groups=grp, cv=gkf, method='predict_proba')[:, 1] for n, mod in models.items()}
pd.DataFrame({n: score(p) for n, p in oof.items()}).T.round(4)

Uncertainty: bootstrap over **students** (not rows) for the AUC of the out-of-fold predictions.

In [ ]:
idx = d.groupby('student_key').indices; ids = list(idx)
rng = np.random.default_rng(RS); res = {n: [] for n in oof if not n.startswith('baseline')}
for _ in range(300):
    r = np.concatenate([idx[i] for i in rng.choice(ids, len(ids))])
    for n in res: res[n].append(roc_auc_score(y.iloc[r], oof[n][r]))
pd.DataFrame({n: {'AUC': roc_auc_score(y, oof[n]), '2.5%': np.percentile(v, 2.5), '97.5%': np.percentile(v, 97.5)} for n, v in res.items()}).T.round(3)

## 3. What carries the signal, and is it calibrated?

In [ ]:
imp = []
for tr, te in gkf.split(X, y, grp):
    mm = gbm(max_depth=3, learning_rate=0.05, max_iter=200).fit(X.iloc[tr], y.iloc[tr])
    imp.append(permutation_importance(mm, X.iloc[te], y.iloc[te], scoring='roc_auc', n_repeats=5, random_state=RS).importances_mean)
print('AUC drop when a feature is shuffled (held-out folds):'); print(pd.Series(np.mean(imp, 0), feat).sort_values(ascending=False).round(4))
lr = models['logistic (all)'].fit(X, y); print('\nstandardised logistic coefficients:'); print(pd.Series(lr[-1].coef_[0], feat).round(3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for n in ['logistic (all)', 'grad. boosting (all)']:
    fp, mp = calibration_curve(y, oof[n], n_bins=10, strategy='quantile'); ax[0].plot(mp, fp, marker='o', label=n)
    ax[1].hist(oof[n], bins=40, alpha=.5, label=n)
ax[0].plot([0, 1], [0, 1], 'k--'); ax[0].set(xlabel='predicted', ylabel='observed pass rate', title='calibration (out-of-fold)'); ax[0].legend()
ax[1].set_title('predicted probabilities'); ax[1].legend(); plt.tight_layout()

## 4. How much would a naive random split flatter us?
Students differ a lot, and `enrol_year` + `school_code` + `prior_score` nearly fingerprint a student, so a flexible model can learn *who* a row belongs to. A row-level KFold lets that leak.

In [ ]:
deep = lambda: gbm(max_depth=6, max_iter=300)
rnd = cross_val_predict(deep(), X, y, cv=KFold(5, shuffle=True, random_state=RS), method='predict_proba')[:, 1]
grd = cross_val_predict(deep(), X, y, groups=grp, cv=gkf, method='predict_proba')[:, 1]
print('deep GBM AUC, random row split   :', round(roc_auc_score(y, rnd), 3))
print('deep GBM AUC, grouped by student :', round(roc_auc_score(y, grd), 3))

## 5. Would student history help?
For a returning student, the pass rate of their *other* sessions is available. That is a different question (known students only), so it is kept separate from the main result. Leave-one-session-out. Caveat: there is no timestamp, so "other sessions" may include later ones; in deployment only earlier sessions exist, so treat this as optimistic. (The leave-one-out construction also has a within-student artifact that works *against* the model, so the gain is not an artifact of that, but the ordering caveat stands.)

In [ ]:
s = d.groupby('student_key').passed_quiz; n = s.transform('size'); tot = s.transform('sum')
d2 = d.assign(loo_rate=((tot - d.passed_quiz) / (n - 1)).where(n > 1)).dropna(subset=['loo_rate'])
y2 = d2.passed_quiz; mk = lambda: gbm(max_depth=3, learning_rate=0.05, max_iter=200)
p_h = cross_val_predict(mk(), d2[feat + ['loo_rate']], y2, groups=d2.student_key, cv=gkf, method='predict_proba')[:, 1]
p_n = cross_val_predict(mk(), d2[feat], y2, groups=d2.student_key, cv=gkf, method='predict_proba')[:, 1]
print('same rows: AUC without history', round(roc_auc_score(y2, p_n), 3), '| with other-session pass rate', round(roc_auc_score(y2, p_h), 3))

## Conclusions
- **Yes, but only weakly, for a new session of an unseen student.** Grouped-by-student CV gives AUC about 0.60 (bootstrap CI roughly 0.59 to 0.61) and a small gain over the baseline in log loss (0.665 vs 0.681) and Brier (0.236 vs 0.244). Better than chance, but far from a reliable per-session predictor.
- **Signal comes from the session itself:** longer sessions go with passing, more hints with failing. `prior_score`, `school_code` and `enrol_year` add essentially nothing. A plain logistic regression does as well as anything; boosting does slightly worse (it overfits).
- **A random row split would be misleading:** a deeper model scores AUC 0.71 with a random split but 0.55 grouped by student, because it learns to recognise individual students.
- **Who the student is matters much more than anything measured here.** Pass rates vary between students far more than chance allows (SD 0.35 vs 0.20 expected). Adding a student's other-session pass rate lifts AUC from 0.58 to 0.85. That only helps for returning students, and with no timestamps I could not confirm only *earlier* sessions would be used. Check with dated data before relying on it.
- **Data quirks to raise with the data owners:** `prior_score` varies slightly within a student (it should be one placement test), and 19 rows exceed 100.
- Caveat: the hints and minutes effects are associations. Hints may be a symptom of struggling, not a cause of failing.